<a href="https://colab.research.google.com/github/whit7990/usports-fb-xml-quickie-stats/blob/main/xml_fb_quickie_stats.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Football Quickie Stats Generator

Turns a **Football Stat Crew / PrestoSports `fbgame` XML** file into a PDF
"Quickie Statistics" page — the cumulative team box score, leader tables,
and scoring summary, generated from wherever the game currently stands in the
file (after Q1, at halftime, final, etc).

<details>
<summary><b>How this works, and accuracy notes (click to expand)</b></summary>

**How it works:** this reads the numbers the XML file *already reports* for
itself — `<venue>` for the header (team names + date), `<team><totals>` for
the team box score (first downs with the rush/pass/penalty breakdown,
rushing, passing, total offense, punts, all return game, penalties, fumbles,
3rd-down conversions, possession time), `<team><player>` for each player's
cumulative stats, and `<scores>`/`<drives>` for the scoring summary (the
receiver on a passing touchdown is looked up from the play-by-play, since
`<scores>` only names the passer). Nothing is recomputed from play-by-play
for the team/player totals, so whatever the file currently contains is
exactly what gets reported — if it was exported right after the 1st quarter
ended, this generates 1st-quarter-only stats; if it's the final file, it
generates final stats.

**Canadian football:** there's no Fourth-Down Conversions row, since Canadian
football only has three downs.

**Tackles for loss:** an unassisted TFL (`tflua`) counts as 1.0, an assisted
TFL (`tfla`) counts as 0.5, matching how the file scores regular tackles. The
pseudo "TEAM" entry (kneel-downs, bad-snap recoveries, etc.) is excluded from
every leader table and never shown with negative passing yards.

**Leader tables:** every category (rushing, passing, receiving, punting,
punt/kick returns, tackles) shows at most the top 4 players, by net yards /
yards / weighted tackles as appropriate. Players are listed as jersey number
+ first initial + last name (e.g. `20 C. Gray-Stapleton`) to save space.

This is an independent, best-effort recreation of the Quickie Stats layout,
not an official Stat Crew tool — spot-check an unfamiliar file's output
against the source XML before relying on it.
</details>


In [1]:
#@title Upload XML file { display-mode: "form" }
import os
xml_path = None

try:
    from google.colab import files
    uploaded = files.upload()
    # Use the first .xml file if several were selected.
    name = next((n for n in uploaded if n.lower().endswith(".xml")), next(iter(uploaded)))
    xml_path = os.path.basename(name)
    with open(xml_path, "wb") as _f:      # write the exact uploaded bytes
        _f.write(uploaded[name])
    print(f"Uploaded: {xml_path} ({len(uploaded[name]):,} bytes)")
    if b"<fbgame" not in uploaded[name][:2000]:
        print("WARNING: this does not look like a Stat Crew fbgame XML file. It starts with:")
        print(uploaded[name][:120])
except ImportError:
    # Not running in Google Colab: set this to your file's path directly.
    xml_path = "vsYork3.xml"
    print(f"Not running in Colab. Using xml_path = {xml_path!r}")


Saving halfvsYork.xml to halfvsYork.xml
Uploaded: halfvsYork.xml (68,653 bytes)


In [2]:
#@title Install dependencies { display-mode: "form" }
!pip install -q reportlab
print("reportlab is ready.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 12.7 MB/s eta 0:00:00
reportlab is ready.


In [3]:
#@title Write parser module (fbxml.py) { display-mode: "form" }
FBXML_SOURCE = r'''"""
Parser for Football Stat Crew (PrestoSports) fbgame XML files.

This reads the file's OWN cumulative team totals (`<team><totals>`) and
per-player cumulative stats (`<team><player>`) directly -- it does not
recompute anything from play-by-play. That means whatever the XML currently
contains is what gets reported: if the file was exported right after the 1st
quarter ended, every number here reflects the game through the 1st quarter;
if it's a final file, everything reflects the final game.

Every figure in the Quickie Statistics box (first downs -- including the
rush/pass/penalty breakdown, rushing, passing, total offense, punts, all
return game, penalties, fumbles, 3rd/4th-down conversions, and possession
time) comes straight from an explicit field in `<totals>`. Nothing is
inferred or guessed.

Tackles-for-loss follow the convention specified by the user: an unassisted
TFL (`tflua`) counts as 1.0, an assisted TFL (`tfla`) counts as 0.5.
"""

import xml.etree.ElementTree as ET
from datetime import datetime


def f(x, default=0):
    """Safe float/int-ish getter for XML attribute strings."""
    try:
        return float(x)
    except (TypeError, ValueError):
        return default


def i(x, default=0):
    try:
        return int(float(x))
    except (TypeError, ValueError):
        return default


def short_name(full_name, uni=None):
    """'Caelan Gray-Stapleton' -> 'C. Gray-Stapleton'; prefixed with jersey number if given."""
    full_name = (full_name or "").strip()
    parts = full_name.split()
    if len(parts) >= 2:
        short = f"{parts[0][0]}. " + " ".join(parts[1:])
    else:
        short = full_name
    return f"{uni} {short}" if uni else short


def format_date(raw):
    """'9/26/2026' -> 'September 26, 2026' (falls back to the raw string)."""
    if not raw:
        return ""
    for fmt in ("%m/%d/%Y", "%m/%d/%y", "%Y-%m-%d"):
        try:
            return datetime.strptime(raw, fmt).strftime("%B %-d, %Y")
        except ValueError:
            continue
    return raw


def load_xml_root(path):
    """Read an fbgame XML file defensively and return its root element.

    Tolerates a UTF-8/UTF-16 byte-order mark, UTF-16 encoding, and stray text
    before the root element, and raises a readable error (with the start of the
    file) if the file isn't really Stat Crew XML -- e.g. an HTML error page or a
    stale/renamed upload.
    """
    with open(path, "rb") as fh:
        raw = fh.read()
    if raw[:2] in (b"\xff\xfe", b"\xfe\xff") or b"\x00" in raw[:40]:
        text = raw.decode("utf-16", errors="replace")
    else:
        try:
            text = raw.decode("utf-8-sig")
        except UnicodeDecodeError:
            text = raw.decode("cp1252", errors="replace")
    start = text.find("<fbgame")
    if start == -1:
        raise ValueError(
            f"{path!r} ({len(raw):,} bytes) does not look like a Football Stat Crew "
            f"fbgame XML file. It starts with: {text[:150]!r}")
    try:
        # Skip the <?xml ...?> declaration so an encoding mismatch can't trip the parser.
        return ET.fromstring(text[start:])
    except ET.ParseError as e:
        raise ValueError(
            f"Could not parse {path!r} ({len(raw):,} bytes): {e}. "
            f"Around the root element it reads: {text[start:start + 150]!r}") from e


ORDINAL = {1: "1st", 2: "2nd", 3: "3rd", 4: "4th", 5: "5th (OT)", 6: "6th (OT)"}


class GameData:
    def __init__(self, xml_path):
        self.root = load_xml_root(xml_path)
        self._parse_meta()
        self._parse_team_totals()
        self._parse_rosters()
        self._parse_scoring()

    # ---------- meta ----------
    def _parse_meta(self):
        venue = self.root.find("venue")
        self.venue = venue.attrib if venue is not None else {}
        self.teams = {}
        for t in self.root.findall("team"):
            vh = t.get("vh")
            linescore = t.find("linescore")
            self.teams[vh] = {
                "id": t.get("id"),
                "name": t.get("name"),
                "record": t.get("record"),
                "vh": vh,
                "score": i(linescore.get("score")) if linescore is not None else None,
            }

        # Figure out how far into the game this file goes, for the page title.
        current_qtr = None
        plays = self.root.find("plays")
        if plays is not None:
            nums = [i(q.get("number")) for q in plays.findall("qtr")]
            if nums:
                current_qtr = max(nums)
        if not current_qtr:
            for t in self.root.findall("team"):
                linescore = t.find("linescore")
                if linescore is not None and linescore.get("line"):
                    parts = [p for p in linescore.get("line").split(",") if p.strip() != ""]
                    current_qtr = max(current_qtr or 0, len(parts))
        self.current_qtr = current_qtr or 1
        self.current_qtr_label = ORDINAL.get(self.current_qtr, f"{self.current_qtr}th")

        # Game clock + completion state, used for the page title.
        # <downtogo qtr=".." clock=".."> at the end of <plays> reports where the
        # game stands; fall back to the last play's clock if it isn't there.
        self.clock = None
        dtg = self.root.findall(".//downtogo")
        if dtg:
            last = dtg[-1]
            if last.get("qtr"):
                self.current_qtr = max(self.current_qtr, i(last.get("qtr")))
            self.clock = last.get("clock") or None
        if not self.clock and plays is not None:
            qtrs = plays.findall("qtr")
            if qtrs and qtrs[-1].findall("play"):
                self.clock = qtrs[-1].findall("play")[-1].get("clock") or None
        status = self.root.find("status")
        self.complete = status is not None and (status.get("complete") or "").upper() == "Y"

    def status_label(self):
        """Title text: 'Halftime', 'Final', or e.g. '4th quarter 01:25'."""
        qtr = self.current_qtr
        at_zero = self.clock in ("00:00", "0:00")
        tied = self.teams["V"]["score"] == self.teams["H"]["score"]
        if at_zero and qtr == 2:
            return "Halftime"
        if qtr >= 4 and (at_zero and not tied or self.complete):
            return "Final"
        name = ORDINAL.get(qtr, f"{qtr}th") + " quarter" if qtr <= 4 else "Overtime"
        return f"{name} {self.clock}" if self.clock else name

    def matchup_header(self):
        """'{away} vs {home} (Month D, YYYY[ at Stadium, Location])', straight from <venue>."""
        v = self.venue
        away = v.get("visname") or self.teams["V"]["name"]
        home = v.get("homename") or self.teams["H"]["name"]
        date_str = format_date(v.get("date"))
        place_bits = [b for b in (v.get("stadium"), v.get("location")) if b]
        place = ", ".join(place_bits)
        paren_bits = [b for b in (date_str, f"at {place}" if place else "") if b]
        paren = f" ({' '.join(paren_bits)})" if paren_bits else ""
        return f"{away} vs {home}{paren}"

    # ---------- team-level cumulative totals ----------
    def _parse_team_totals(self):
        self.totals = {}
        for t in self.root.findall("team"):
            vh = t.get("vh")
            tot_el = t.find("totals")
            d = {}
            if tot_el is not None:
                d["totoff_plays"] = i(tot_el.get("totoff_plays"))
                d["totoff_yards"] = i(tot_el.get("totoff_yards"))
                for child in tot_el:
                    d[child.tag] = {k: v for k, v in child.attrib.items()}
            self.totals[vh] = d

        # "Team Losses" -- the pseudo "TEAM" player's rushing yardage
        # (kneel-downs, bad-snap recoveries, etc. not credited to a real player).
        self.team_losses = {"V": 0, "H": 0}
        for t in self.root.findall("team"):
            vh = t.get("vh")
            for p in t.findall("player"):
                if (p.get("uni") or "").upper() == "TM":
                    rush = p.find("rush")
                    if rush is not None:
                        self.team_losses[vh] += i(rush.get("yds"))

    # ---------- per-player cumulative stats ----------
    def _parse_rosters(self):
        self.players = {"V": [], "H": []}
        self.uni_to_name = {"V": {}, "H": {}}
        for t in self.root.findall("team"):
            vh = t.get("vh")
            for p in t.findall("player"):
                name = p.get("name") or ""
                is_team = (p.get("uni") or "").upper() == "TM" or name.strip().upper() == "TEAM"
                entry = {"name": name, "uni": p.get("uni"), "is_team": is_team}
                for tag in ("rush", "pass", "rcv", "punt", "pr", "kr", "ir", "defense", "fumbles", "fg", "pat"):
                    el = p.find(tag)
                    if el is not None:
                        entry[tag] = {k: v for k, v in el.attrib.items()}
                self.players[vh].append(entry)
                if p.get("uni"):
                    self.uni_to_name[vh][p.get("uni")] = name

    # ---------- scoring plays, for the scoring summary table ----------
    def _parse_scoring(self):
        self.drives_by_index = {}
        drives = self.root.find("drives")
        if drives is not None:
            for d in drives.findall("drive"):
                self.drives_by_index[d.get("driveindex")] = d.attrib

        self.scores = []
        scores_el = self.root.find("scores")
        if scores_el is not None:
            for s in scores_el.findall("score"):
                self.scores.append(dict(s.attrib))

        # Look up the receiver for every passing touchdown (the <scores> block
        # only names the passer as "scorer"; the receiver has to be found in
        # the play-by-play's <p_pa rcv="..."> for the matching scoring play).
        self.pass_td_receiver = {}
        plays_root = self.root.find("plays")
        if plays_root is not None:
            for qtr_el in plays_root.findall("qtr"):
                qnum = qtr_el.get("number")
                for play in qtr_el.findall("play"):
                    text = (play.get("text") or "").upper()
                    if play.get("type") != "P" or "TOUCHDOWN" not in text:
                        continue
                    for child in play:
                        if child.tag == "p_pa" and child.get("result") == "COMP":
                            vh = child.get("vh")
                            rcv_uni = child.get("rcv")
                            receiver = self.uni_to_name.get(vh, {}).get(rcv_uni)
                            if receiver:
                                key = (qnum, play.get("clock"), self.teams[vh]["id"])
                                self.pass_td_receiver[key] = receiver

    def receiver_for_score(self, score):
        """Look up the receiver name for a passing-touchdown <score> entry."""
        key = (score.get("qtr"), score.get("clock"), score.get("team"))
        return self.pass_td_receiver.get(key)

    # ---------- convenience accessors used by the renderer ----------
    def score(self, vh):
        return self.teams[vh]["score"]

    def totals_sub(self, vh, tag, attr, default=0):
        return i(self.totals.get(vh, {}).get(tag, {}).get(attr), default)

    def pass_yds(self, vh):
        """Net passing yards; guarded against dipping below 0 (a team's total
        net passing yardage is reported as 0 rather than negative, matching
        standard box-score convention, even if sack yardage would put the raw
        figure below zero)."""
        return max(0, self.totals_sub(vh, "pass", "yds"))

    def top(self, vh):
        """Possession time, 'MM:SS', as reported by the file."""
        return self.totals.get(vh, {}).get("misc", {}).get("top", "00:00")

'''

with open("fbxml.py", "w") as _f:
    _f.write(FBXML_SOURCE)
print("Wrote fbxml.py")


Wrote fbxml.py


In [4]:
#@title Write PDF renderer module (quickie_pdf.py) { display-mode: "form" }
QUICKIE_PDF_SOURCE = r'''"""
Render a single "Quickie Statistics" page (current cumulative state of the
game, as of whatever the XML file currently contains) to a PDF.
"""

from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER

from fbxml import i, f, short_name

FONT = "Courier"
FONT_B = "Courier-Bold"
FONT_I = "Courier-Oblique"

MAX_PER_CATEGORY = 4

REPO_URL = "https://github.com/whit7990/usports-fb-xml-quickie-stats"


def fmt_avg(yds, no):
    return f"{yds / no:.1f}" if no else "0.0"


def player_label(p):
    return short_name(p["name"], p.get("uni"))


def build_team_box(game):
    v_id, h_id = game.teams["V"]["id"], game.teams["H"]["id"]

    def row(label, vval, hval):
        return [label, vval, hval]

    # Row indices, for the style rules below:
    #  0 header, 1 Score, 2 FIRST DOWNS, 3 Rushing, 4 Passing, 5 Penalty,
    #  6 RUSHES-YARDS, 7 PASSING YDS, 8 Passes Att-Comp-Int, 9 Team Losses,
    #  10 TOTAL OFFENSE, 11 Fumble Ret, 12 Punt Ret, 13 KO Ret, 14 Int Ret,
    #  15 Punts, 16 Fumbles-Lost, 17 Penalties, 18 Possession, 19 3rd down
    data = [["", v_id, h_id]]
    data.append(row("Score", game.score("V"), game.score("H")))
    data.append(row("FIRST DOWNS",
                     game.totals_sub("V", "firstdowns", "no"),
                     game.totals_sub("H", "firstdowns", "no")))
    data.append(row("  Rushing",
                     game.totals_sub("V", "firstdowns", "rush"),
                     game.totals_sub("H", "firstdowns", "rush")))
    data.append(row("  Passing",
                     game.totals_sub("V", "firstdowns", "pass"),
                     game.totals_sub("H", "firstdowns", "pass")))
    data.append(row("  Penalty",
                     game.totals_sub("V", "firstdowns", "penalty"),
                     game.totals_sub("H", "firstdowns", "penalty")))
    data.append(row("RUSHES-YARDS (NET)",
                     f'{game.totals_sub("V","rush","att")}-{game.totals_sub("V","rush","yds")}',
                     f'{game.totals_sub("H","rush","att")}-{game.totals_sub("H","rush","yds")}'))
    data.append(row("PASSING YDS (NET)", game.pass_yds("V"), game.pass_yds("H")))
    data.append(row("Passes Att-Comp-Int",
                     f'{game.totals_sub("V","pass","att")}-{game.totals_sub("V","pass","comp")}-{game.totals_sub("V","pass","int")}',
                     f'{game.totals_sub("H","pass","att")}-{game.totals_sub("H","pass","comp")}-{game.totals_sub("H","pass","int")}'))
    data.append(row("Team Losses", game.team_losses["V"], game.team_losses["H"]))
    data.append(row("TOTAL OFFENSE PLAYS-YARDS",
                     f'{game.totals["V"].get("totoff_plays", 0)}-{game.totals["V"].get("totoff_yards", 0)}',
                     f'{game.totals["H"].get("totoff_plays", 0)}-{game.totals["H"].get("totoff_yards", 0)}'))
    data.append(row("Fumble Returns-Yards",
                     f'{game.totals_sub("V","defense","fr")}-{game.totals_sub("V","defense","fryds")}',
                     f'{game.totals_sub("H","defense","fr")}-{game.totals_sub("H","defense","fryds")}'))
    data.append(row("Punt Returns-Yards",
                     f'{game.totals_sub("V","pr","no")}-{game.totals_sub("V","pr","yds")}',
                     f'{game.totals_sub("H","pr","no")}-{game.totals_sub("H","pr","yds")}'))
    data.append(row("Kickoff Returns-Yards",
                     f'{game.totals_sub("V","kr","no")}-{game.totals_sub("V","kr","yds")}',
                     f'{game.totals_sub("H","kr","no")}-{game.totals_sub("H","kr","yds")}'))
    data.append(row("Interception Returns-Yards",
                     f'{game.totals_sub("V","ir","no")}-{game.totals_sub("V","ir","yds")}',
                     f'{game.totals_sub("H","ir","no")}-{game.totals_sub("H","ir","yds")}'))
    data.append(row("Punts (Number-Avg)",
                     f'{game.totals_sub("V","punt","no")}-{game.totals["V"].get("punt",{}).get("avg","0.0")}',
                     f'{game.totals_sub("H","punt","no")}-{game.totals["H"].get("punt",{}).get("avg","0.0")}'))
    data.append(row("Fumbles-Lost",
                     f'{game.totals_sub("V","fumbles","no")}-{game.totals_sub("V","fumbles","lost")}',
                     f'{game.totals_sub("H","fumbles","no")}-{game.totals_sub("H","fumbles","lost")}'))
    data.append(row("Penalties-Yards",
                     f'{game.totals_sub("V","penalties","no")}-{game.totals_sub("V","penalties","yds")}',
                     f'{game.totals_sub("H","penalties","no")}-{game.totals_sub("H","penalties","yds")}'))
    data.append(row("Possession Time", game.top("V"), game.top("H")))
    data.append(row("Third-Down Conversions",
                     f'{game.totals_sub("V","conversions","thirdconv")} of {game.totals_sub("V","conversions","thirdatt")}',
                     f'{game.totals_sub("H","conversions","thirdconv")} of {game.totals_sub("H","conversions","thirdatt")}'))
    # Note: no Fourth-Down Conversions row -- Canadian football has 3 downs.

    t = Table(data, colWidths=[2.6 * inch, 1.6 * inch, 1.6 * inch])
    style = [
        ("FONT", (0, 0), (-1, -1), FONT, 8),
        ("FONT", (0, 0), (-1, 0), FONT_B, 8),
        ("FONT", (1, 1), (1, 1), FONT_B, 8),
        ("ALIGN", (1, 0), (-1, -1), "CENTER"),
        ("LINEBELOW", (0, 0), (-1, 0), 1, colors.black),
        ("LINEBELOW", (0, -1), (-1, -1), 0.5, colors.grey),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.whitesmoke, colors.white]),
        ("TOPPADDING", (0, 0), (-1, -1), 0),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 0),
        ("LEADING", (0, 0), (-1, -1), 9),
    ]
    # Bold the headline rows.
    for r in (2, 6, 7, 10):
        style.append(("FONT", (0, r), (-1, r), FONT_B, 8))
    # Italicize the first-down breakdown sub-rows (normal color, no grey).
    for r in (3, 4, 5):
        style.append(("FONT", (0, r), (0, r), FONT_I, 7.5))
    t.setStyle(TableStyle(style))
    return t


def leader_table(rows, headers, colw):
    """A leader table with just a bold header row (no separate spanning title)."""
    data = [headers] + rows
    t = Table(data, colWidths=colw)
    t.setStyle(TableStyle([
        ("FONT", (0, 0), (-1, 0), FONT_B, 7.5),
        ("FONT", (0, 1), (-1, -1), FONT, 7.5),
        ("ALIGN", (1, 0), (-1, -1), "CENTER"),
        ("ALIGN", (0, 0), (0, 0), "LEFT"),
        ("LINEBELOW", (0, 0), (-1, 0), 0.75, colors.black),
        ("TOPPADDING", (0, 0), (-1, -1), 0),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 0),
        ("LEADING", (0, 0), (-1, -1), 8.5),
    ]))
    return t


def top_n(players, tag, sort_key, n=MAX_PER_CATEGORY, exclude_team=True):
    entries = [p for p in players if tag in p and (not exclude_team or not p["is_team"])]
    entries.sort(key=lambda p: f(p[tag].get(sort_key, 0)), reverse=True)
    return entries[:n]


def team_leaders(game, vh):
    players = game.players[vh]
    flow = []

    rows = []
    for p in top_n(players, "rush", "yds"):
        r = p["rush"]
        att = i(r.get("att"))
        net = i(r.get("yds"))
        rows.append([player_label(p), att, i(r.get("gain")), i(r.get("loss")), net, i(r.get("td")), i(r.get("long")),
                     fmt_avg(net, att)])
    if rows:
        flow.append(leader_table(rows, ["Rushing", "No.", "Gain", "Loss", "Net", "TD", "Lg", "Avg"],
                                  [1.35 * inch] + [0.29 * inch] * 7))
        flow.append(Spacer(1, 3))

    rows = []
    for p in top_n(players, "pass", "yds"):
        pa = p["pass"]
        rows.append([player_label(p), f'{i(pa.get("comp"))}-{i(pa.get("att"))}-{i(pa.get("int"))}',
                     i(pa.get("yds")), i(pa.get("td")), i(pa.get("int")), i(pa.get("long")), i(pa.get("sacks"))])
    if rows:
        flow.append(leader_table(rows, ["Passing", "C-A-I", "Yds", "TD", "Int", "Long", "Sack"],
                                  [1.05 * inch] + [0.42 * inch] * 6))
        flow.append(Spacer(1, 3))

    rows = []
    for p in top_n(players, "rcv", "yds"):
        r = p["rcv"]
        rows.append([player_label(p), i(r.get("no")), i(r.get("yds")), i(r.get("td")), i(r.get("long"))])
    if rows:
        flow.append(leader_table(rows, ["Receiving", "No.", "Yards", "TD", "Long"],
                                  [1.3 * inch] + [0.58 * inch] * 4))
        flow.append(Spacer(1, 3))

    rows = []
    for p in top_n(players, "punt", "yds"):
        r = p["punt"]
        rows.append([player_label(p), i(r.get("no")), i(r.get("yds")), r.get("avg", "0.0"), i(r.get("long"))])
    if rows:
        flow.append(leader_table(rows, ["Punting", "No", "Yds", "Avg", "Long"],
                                  [1.3 * inch] + [0.58 * inch] * 4))
        flow.append(Spacer(1, 3))

    rows = []
    for p in top_n(players, "pr", "yds"):
        r = p["pr"]
        rows.append([player_label(p), i(r.get("no")), i(r.get("yds")), i(r.get("long"))])
    if rows:
        flow.append(leader_table(rows, ["Punt Returns", "No.", "Yards", "Long"],
                                  [1.5 * inch] + [0.68 * inch] * 3))
        flow.append(Spacer(1, 3))

    rows = []
    for p in top_n(players, "kr", "yds"):
        r = p["kr"]
        rows.append([player_label(p), i(r.get("no")), i(r.get("yds")), i(r.get("long"))])
    if rows:
        flow.append(leader_table(rows, ["Kick Returns", "No.", "Yards", "Long"],
                                  [1.5 * inch] + [0.68 * inch] * 3))
        flow.append(Spacer(1, 3))

    # Tackles: sort by (unassisted + 0.5*assisted), same weighting as TFL per the user's spec
    def_players = [p for p in players if "defense" in p and not p["is_team"]]

    def weighted_total(p):
        d = p["defense"]
        return i(d.get("tackua")) + 0.5 * i(d.get("tacka"))

    def_players.sort(key=weighted_total, reverse=True)
    rows = []
    for p in def_players[:MAX_PER_CATEGORY]:
        d = p["defense"]
        ua, a = i(d.get("tackua")), i(d.get("tacka"))
        total = ua + 0.5 * a
        sacks = d.get("sacksua") or d.get("sackua") or 0  # naming varies between team/player level
        tfl_ua, tfl_a = i(d.get("tflua")), i(d.get("tfla"))
        tfl_total = tfl_ua + 0.5 * tfl_a
        rows.append([player_label(p), f'{ua}-{a}', f'{total:g}', i(sacks), f'{tfl_total:g}', i(d.get("brup"))])
    if rows:
        flow.append(leader_table(rows, ["Tackles", "UA-A", "Total", "Sacks", "TFL", "BrUp"],
                                  [1.25 * inch] + [0.48 * inch] * 5))

    return flow


# ---------- scoring summary (bottom of page) ----------

PLAY_LABEL = {"field goal": "field goal", "rush": "run", "pass": "pass", "safety": "safety"}


def _pat_str(s):
    patby, patres, pattype = s.get("patby"), s.get("patres"), s.get("pattype")
    if not patres:
        return ""
    verb = {"KICK": "kick", "PASS": "pass", "RUSH": "run"}.get((pattype or "").upper(), (pattype or "").lower())
    if patres.upper() == "GOOD":
        return f" ({patby} {verb})" if patby else f" ({verb} good)"
    return f" ({patby} {verb} failed)" if patby else f" ({verb} failed)"


def scoring_rows(game):
    rows = []
    for s in game.scores:
        qtr = s.get("qtr", "")
        clock = s.get("clock", "")
        team = s.get("team", "")
        scorer = s.get("scorer", "")
        how = (s.get("how") or "").lower()
        yds = i(s.get("yds"))
        label = PLAY_LABEL.get(how, how)

        drive = game.drives_by_index.get(s.get("driveindex"), {})
        plays = drive.get("plays", "")
        drive_yds = drive.get("yards", "")
        top = s.get("top", "")

        if how == "safety":
            play_desc = f"{team} - {scorer} safety"
        elif how == "pass":
            receiver = game.receiver_for_score(s)
            to_whom = f" to {receiver}" if receiver else ""
            play_desc = f"{team} - {scorer} {yds} yd pass{to_whom}{_pat_str(s)}"
        else:
            play_desc = f"{team} - {scorer} {yds} yd {label}{_pat_str(s)}"
        if plays and drive_yds:
            play_desc += f", {plays}-{drive_yds} {top}"

        vscore, hscore = s.get("vscore", ""), s.get("hscore", "")
        rows.append([qtr, clock, play_desc, f"{vscore} - {hscore}"])
    return rows


def build_scoring_table(game):
    rows = scoring_rows(game)
    if not rows:
        return None
    v_id, h_id = game.teams["V"]["id"], game.teams["H"]["id"]
    header = ["Qtr", "Time", "Scoring Play", f"{v_id}-{h_id}"]
    data = [header] + rows
    t = Table(data, colWidths=[0.4 * inch, 0.55 * inch, 5.85 * inch, 0.7 * inch])
    t.setStyle(TableStyle([
        ("FONT", (0, 0), (-1, 0), FONT_B, 7.5),
        ("FONT", (0, 1), (-1, -1), FONT, 7.5),
        ("ALIGN", (0, 0), (1, -1), "CENTER"),
        ("ALIGN", (3, 0), (3, -1), "CENTER"),
        ("LINEBELOW", (0, 0), (-1, 0), 0.75, colors.black),
        ("TOPPADDING", (0, 0), (-1, -1), 1),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 1),
        ("LEADING", (0, 0), (-1, -1), 8.5),
    ]))
    return t


def render_quickie_pdf(game, out_path, game_title=None):
    styles = getSampleStyleSheet()
    title_style = ParagraphStyle("QTitle", parent=styles["Heading2"], alignment=TA_CENTER,
                                  fontName=FONT_B, fontSize=15, leading=18)
    sub_style = ParagraphStyle("QSub", parent=styles["Normal"], alignment=TA_CENTER,
                                fontName=FONT_B, fontSize=12, leading=15)
    name_style = ParagraphStyle("QName", parent=styles["Normal"], fontName=FONT_B, fontSize=9)

    doc = SimpleDocTemplate(out_path, pagesize=letter,
                             topMargin=0.35 * inch, bottomMargin=0.5 * inch,
                             leftMargin=0.5 * inch, rightMargin=0.5 * inch)

    v_name, h_name = game.teams["V"]["name"], game.teams["H"]["name"]
    matchup = game_title or game.matchup_header()

    story = [
        Paragraph(f"Quickie Statistics ({game.status_label()})", title_style),
        Spacer(1, 2),
        Paragraph(matchup, sub_style),
        Spacer(1, 4),
        build_team_box(game),
        Spacer(1, 5),
    ]

    left = [Paragraph(v_name, name_style), Spacer(1, 2)] + team_leaders(game, "V")
    right = [Paragraph(h_name, name_style), Spacer(1, 2)] + team_leaders(game, "H")

    side_by_side = Table([[left, right]], colWidths=[3.75 * inch, 3.75 * inch])
    side_by_side.setStyle(TableStyle([
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 2),
        ("RIGHTPADDING", (0, 0), (-1, -1), 2),
    ]))
    story.append(side_by_side)

    scoring_table = build_scoring_table(game)
    if scoring_table is not None:
        story.append(Spacer(1, 6))
        story.append(Paragraph("Scoring Summary", name_style))
        story.append(Spacer(1, 2))
        story.append(scoring_table)

    def draw_footer(canvas, doc_):
        canvas.saveState()
        canvas.setFont("Helvetica", 8)
        canvas.setFillColor(colors.black)
        canvas.drawString(doc_.leftMargin, 0.3 * inch, REPO_URL)
        canvas.restoreState()

    doc.build(story, onFirstPage=draw_footer, onLaterPages=draw_footer)
    return out_path
'''

with open("quickie_pdf.py", "w") as _f:
    _f.write(QUICKIE_PDF_SOURCE)
print("Wrote quickie_pdf.py")


Wrote quickie_pdf.py


In [5]:
#@title Parse game { display-mode: "form" }
from fbxml import GameData

game = GameData(xml_path)

v, h = game.teams["V"], game.teams["H"]
print(game.matchup_header())
print(f"Game status: {game.status_label()}  |  {v['id']} {v['score']} - {h['id']} {h['score']}")


York vs Laurier (September 26, 2026)
Game status: Halftime  |  YRK 11 - WLU 16


In [6]:
#@title Generate PDF { display-mode: "form" }
import os
from quickie_pdf import render_quickie_pdf

out_path = os.path.splitext(os.path.basename(xml_path))[0] + ".pdf"
render_quickie_pdf(game, out_path)
print(f"Wrote {out_path}")

try:
    from google.colab import files
    files.download(out_path)
except ImportError:
    print(f"Saved to {out_path} -- find it in your working directory.")


Wrote halfvsYork.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>